# 03 · Full receipt-extraction pipeline (standalone)

Upload **this notebook only** to Colab or Jupyter. It does not import `app`, `training`, `eval`, or `scripts`.

**Colab:** Runtime → GPU (T4) → run all. CPU machines can run every cell up to the VLM section.

Pipeline in this notebook:

1. Map CORD labels to the assignment JSON schema
2. Parse / repair model JSON
3. Ground every non-null leaf against page text (null-out hallucinations)
4. Merge pages, then sanity-check totals
5. Zero-shot eval on a frozen CORD test slice
6. QLoRA fine-tune Qwen2.5-VL-3B (Unsloth, else peft+trl+bnb)
7. Fine-tuned eval on the **same** test ids
8. Write metric tables (and wandb when a key is set)

Model: `Qwen/Qwen2.5-VL-3B-Instruct`, QLoRA 4-bit, LoRA r=16 α=32, language layers only. Resolution: long-edge 1008 / `max_pixels = 1008×28×28`.


## 0. Setup

The install cell is a no-op outside Colab. On a T4, uncomment is not needed — Colab is detected.


In [ ]:
import hashlib
import json
import math
import os
import re
import sys
import time
from collections import defaultdict
from datetime import datetime
from io import BytesIO
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic(
        "pip",
        "install -q datasets transformers peft trl bitsandbytes accelerate wandb pillow json-repair huggingface_hub",
    )
    get_ipython().system("pip install -q unsloth || true")

os.environ.setdefault(
    "WANDB_MODE",
    "disabled" if not os.environ.get("WANDB_API_KEY") else os.environ.get("WANDB_MODE", "online"),
)

try:
    import torch

    HAS_GPU = torch.cuda.is_available()
except Exception:
    torch = None
    HAS_GPU = False

MODEL_ID = os.environ.get("MODEL_ID", "Qwen/Qwen2.5-VL-3B-Instruct")
ADAPTER_DIR = Path(os.environ.get("ADAPTER_DIR", "outputs/lora"))
HF_ADAPTER_REPO = os.environ.get("HF_ADAPTER_REPO", "mahesh-nallada/vlm-receipt-extraction-lora")
LONG_EDGE = 1008
MAX_PIXELS = 1008 * 28 * 28
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EVAL_N = int(os.environ.get("EVAL_N", "16"))
SANITY_TOLERANCE = 0.05
WORK = Path("pipeline_out")
WORK.mkdir(exist_ok=True)
print("colab", IN_COLAB, "cuda", HAS_GPU, "model", MODEL_ID)


## 1. Schema, CORD mapping, JSON parse

CORD has no store-name or date labels. Those targets are `null`. Dates in the live API are ISO `YYYY-MM-DD`. Numbers are JSON numbers.


In [ ]:
EXTRACT_PROMPT = (
    "Extract the receipt as JSON with keys store_name, date (YYYY-MM-DD), "
    "line_items (name, qty, unit_price, amount), subtotal, tax, total. "
    "Copy only text visible on the page. Use null when a field is absent. "
    "Numbers must be JSON numbers, not strings. Do not guess."
)

QTY_RE = re.compile(r"[-+]?\d+(?:[.,]\d+)?")
FENCE = re.compile(r"```(?:json)?\s*(.*?)```", re.DOTALL)


def parse_qty(raw):
    if raw is None:
        return None
    if isinstance(raw, (int, float)) and not isinstance(raw, bool):
        return float(raw)
    match = QTY_RE.search(str(raw))
    return float(match.group(0).replace(",", ".")) if match else None


def parse_money(raw):
    if raw is None:
        return None
    if isinstance(raw, (int, float)) and not isinstance(raw, bool):
        return float(raw)
    text = str(raw).strip().replace("Rp", "").replace("rp", "").replace(" ", "")
    if not text:
        return None
    if "," in text and "." in text:
        text = text.replace(".", "").replace(",", ".") if text.rfind(",") > text.rfind(".") else text.replace(",", "")
    elif "," in text:
        parts = text.split(",")
        text = "".join(parts) if len(parts[-1]) == 3 else text.replace(",", ".")
    try:
        return float(text)
    except ValueError:
        return None


def as_menu_list(menu):
    if menu is None:
        return []
    if isinstance(menu, dict):
        return [menu]
    return [item for item in menu if isinstance(item, dict)] if isinstance(menu, list) else []


def cord_to_schema(gt_parse):
    items = []
    for item in as_menu_list(gt_parse.get("menu")):
        name = item.get("nm")
        if not isinstance(name, str) or not name.strip():
            continue
        items.append(
            {
                "name": name.strip(),
                "qty": parse_qty(item.get("cnt")),
                "unit_price": parse_money(item.get("unitprice")),
                "amount": parse_money(item.get("price")),
            }
        )
    sub = gt_parse.get("sub_total") or {}
    tot = gt_parse.get("total") or {}
    return {
        "store_name": None,
        "date": None,
        "line_items": items,
        "subtotal": parse_money(sub.get("subtotal_price")),
        "tax": parse_money(sub.get("tax_price")),
        "total": parse_money(tot.get("total_price")),
    }


def ground_truth_of(row):
    raw = row["ground_truth"]
    payload = json.loads(raw) if isinstance(raw, str) else raw
    return payload["gt_parse"]


def split_hash(ids):
    return hashlib.sha256(",".join(ids).encode()).hexdigest()[:16]


def extract_json_object(text):
    fenced = FENCE.search(text)
    candidate = fenced.group(1) if fenced else text
    start, end = candidate.find("{"), candidate.rfind("}")
    if start == -1 or end <= start:
        return None
    return candidate[start : end + 1]


def parse_model_output(text):
    blob = extract_json_object(text or "")
    if blob is None:
        raise ValueError("model output is not JSON")
    try:
        loaded = json.loads(blob)
    except json.JSONDecodeError:
        try:
            from json_repair import repair_json

            loaded = json.loads(repair_json(blob))
        except Exception as exc:
            raise ValueError("model output is not valid JSON") from exc
    if not isinstance(loaded, dict):
        raise ValueError("model output must be an object")
    items = []
    for item in loaded.get("line_items") or []:
        if not isinstance(item, dict) or not str(item.get("name") or "").strip():
            continue
        items.append(
            {
                "name": str(item["name"]).strip(),
                "qty": parse_qty(item.get("qty")),
                "unit_price": parse_money(item.get("unit_price")),
                "amount": parse_money(item.get("amount")),
            }
        )
    date = loaded.get("date")
    if date is not None:
        datetime.strptime(str(date), "%Y-%m-%d")
    store = loaded.get("store_name")
    return {
        "store_name": store.strip() if isinstance(store, str) and store.strip() else None,
        "date": date,
        "line_items": items,
        "subtotal": parse_money(loaded.get("subtotal")),
        "tax": parse_money(loaded.get("tax")),
        "total": parse_money(loaded.get("total")),
    }


print("schema helpers ready")


## 2. Grounding, sanity, merge

**Null-out:** a predicted value not found in the document text becomes `null` + `not_found`. Empty text (no layer, no OCR) → `null` + `unverified`.

**Sanity:** failed sums keep the numbers (they were on the page) and mark those money fields `unverified`. Tolerance 0.05.

**Merge:** concatenate line items. Header fields from the page with more verified `store_name`/`date` (ties: first page). Totals from the page with more verified money leaves (ties: last page).


In [ ]:
CURRENCY = re.compile(r"(?:[$€£¥₹]|rp\.?|idr|usd|eur|gbp)\s*", re.I)
SPACES = re.compile(r"\s+")


def normalize_text(value):
    text = CURRENCY.sub("", str(value).casefold().replace("\u00a0", " "))
    return SPACES.sub(" ", text).strip()


def normalize_number_token(value):
    text = normalize_text(value).replace(" ", "")
    if not text:
        return ""
    if "," in text and "." in text:
        text = text.replace(".", "").replace(",", ".") if text.rfind(",") > text.rfind(".") else text.replace(",", "")
    elif "," in text:
        parts = text.split(",")
        text = "".join(parts) if len(parts[-1]) == 3 and parts[0].lstrip("-").isdigit() else text.replace(",", ".")
    return text


def parse_number(value):
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return float(value)
    token = normalize_number_token(value)
    try:
        return float(token) if token else None
    except ValueError:
        return None


def numeric_variants(value):
    if abs(value - round(value)) < 1e-9:
        whole = str(int(round(value)))
        grouped = f"{int(round(value)):,}"
        return [whole, grouped, grouped.replace(",", ".")]
    two = f"{value:.2f}"
    integer, frac = two.split(".")
    return [
        two,
        two.rstrip("0").rstrip("."),
        two.replace(".", ","),
        f"{int(integer):,}.{frac}",
        f"{int(integer):,}".replace(",", ".") + f",{frac}",
    ]


def numbers_match(predicted, document_text, tolerance=SANITY_TOLERANCE):
    hay = normalize_text(document_text)
    compact = hay.replace(",", "").replace(" ", "")
    if any(v and (v in hay or v in compact) for v in numeric_variants(predicted)):
        return True
    for token in re.findall(r"[-+]?\d[\d.,]*", document_text):
        parsed = parse_number(token)
        if parsed is not None and abs(parsed - predicted) <= tolerance:
            return True
    return False


def text_present(predicted, document_text):
    needle = normalize_text(predicted)
    hay = normalize_text(document_text)
    return bool(needle) and (needle in hay or needle.replace(" ", "") in hay.replace(" ", ""))


def date_present(iso, document_text):
    year, month, day = iso.split("-")
    month_i, day_i = int(month), int(day)
    candidates = {
        iso,
        f"{day}/{month}/{year}",
        f"{month}/{day}/{year}",
        f"{day}-{month}-{year}",
        f"{day_i}/{month_i}/{year}",
        f"{month_i}/{day_i}/{year}",
    }
    hay = normalize_text(document_text)
    return any(normalize_text(c) in hay for c in candidates)


def verification_paths(n):
    paths = ["store_name", "date"]
    for i in range(n):
        paths.extend([f"line_items.{i}.name", f"line_items.{i}.qty", f"line_items.{i}.unit_price", f"line_items.{i}.amount"])
    return paths + ["subtotal", "tax", "total"]


def _status(value, present, empty):
    if value is None:
        return None, "not_found"
    if empty:
        return None, "unverified"
    return (value, "verified") if present else (None, "not_found")


def ground_receipt(receipt, document_text):
    empty = not normalize_text(document_text)
    store, store_s = _status(receipt.get("store_name"), text_present(receipt.get("store_name") or "", document_text), empty)
    date, date_s = _status(
        receipt.get("date"),
        date_present(receipt["date"], document_text) if receipt.get("date") else False,
        empty,
    )
    items, ver = [], {"store_name": store_s, "date": date_s}
    for item in receipt.get("line_items") or []:
        name, name_s = _status(item.get("name"), text_present(item.get("name") or "", document_text), empty)
        if name is None:
            continue
        qty, qty_s = _status(item.get("qty"), numbers_match(item["qty"], document_text) if item.get("qty") is not None else False, empty)
        unit, unit_s = _status(
            item.get("unit_price"),
            numbers_match(item["unit_price"], document_text) if item.get("unit_price") is not None else False,
            empty,
        )
        amt, amt_s = _status(
            item.get("amount"),
            numbers_match(item["amount"], document_text) if item.get("amount") is not None else False,
            empty,
        )
        idx = len(items)
        items.append({"name": name, "qty": qty, "unit_price": unit, "amount": amt})
        ver[f"line_items.{idx}.name"] = name_s
        ver[f"line_items.{idx}.qty"] = qty_s
        ver[f"line_items.{idx}.unit_price"] = unit_s
        ver[f"line_items.{idx}.amount"] = amt_s
    sub, sub_s = _status(receipt.get("subtotal"), numbers_match(receipt["subtotal"], document_text) if receipt.get("subtotal") is not None else False, empty)
    tax, tax_s = _status(receipt.get("tax"), numbers_match(receipt["tax"], document_text) if receipt.get("tax") is not None else False, empty)
    tot, tot_s = _status(receipt.get("total"), numbers_match(receipt["total"], document_text) if receipt.get("total") is not None else False, empty)
    ver.update({"subtotal": sub_s, "tax": tax_s, "total": tot_s})
    return {"store_name": store, "date": date, "line_items": items, "subtotal": sub, "tax": tax, "total": tot, "_verification": ver}


def check_totals(receipt, tolerance=SANITY_TOLERANCE):
    ver = dict(receipt.get("_verification") or {})
    amounts = [i["amount"] for i in receipt.get("line_items") or [] if i.get("amount") is not None]
    items_total = sum(amounts) if amounts else None
    sub, tax, tot = receipt.get("subtotal"), receipt.get("tax"), receipt.get("total")
    if items_total is not None and sub is not None and abs(items_total - sub) > tolerance:
        ver["subtotal"] = "unverified"
        for i, item in enumerate(receipt.get("line_items") or []):
            if item.get("amount") is not None:
                ver[f"line_items.{i}.amount"] = "unverified"
    if sub is not None and tot is not None and abs(sub + (tax or 0.0) - tot) > tolerance:
        ver["subtotal"] = ver["total"] = "unverified"
        if tax is not None:
            ver["tax"] = "unverified"
    out = dict(receipt)
    out["_verification"] = ver
    return out


def page_score(receipt, fields):
    ver = receipt.get("_verification") or {}
    return sum(1 for f in fields if ver.get(f) == "verified")


def merge_pages(pages):
    if not pages:
        return {"store_name": None, "date": None, "line_items": [], "subtotal": None, "tax": None, "total": None,
                "_verification": {k: "not_found" for k in verification_paths(0)}}
    if len(pages) == 1:
        return pages[0]
    items, item_ver = [], {}
    for page in pages:
        offset = len(items)
        items.extend(page.get("line_items") or [])
        for i, _ in enumerate(page.get("line_items") or []):
            for leaf in ("name", "qty", "unit_price", "amount"):
                item_ver[f"line_items.{offset + i}.{leaf}"] = page["_verification"][f"line_items.{i}.{leaf}"]
    header = max(enumerate(pages), key=lambda row: (page_score(row[1], ("store_name", "date")), -row[0]))[1]
    money = max(enumerate(pages), key=lambda row: (page_score(row[1], ("subtotal", "tax", "total")), row[0]))[1]
    ver = {**item_ver, "store_name": header["_verification"]["store_name"], "date": header["_verification"]["date"],
           "subtotal": money["_verification"]["subtotal"], "tax": money["_verification"]["tax"], "total": money["_verification"]["total"]}
    for path in verification_paths(len(items)):
        ver.setdefault(path, "not_found")
    return {"store_name": header.get("store_name"), "date": header.get("date"), "line_items": items,
            "subtotal": money.get("subtotal"), "tax": money.get("tax"), "total": money.get("total"), "_verification": ver}


print("ground / sanity / merge ready")


## 3. Extract pipeline

One function: raw model text per page → parse (one repair retry) → ground per page → merge → ground on full text → sanity.


In [ ]:
def empty_receipt():
    return {
        "store_name": None,
        "date": None,
        "line_items": [],
        "subtotal": None,
        "tax": None,
        "total": None,
        "_verification": {k: "not_found" for k in verification_paths(0)},
    }


def extract_one_page(raw_text, repair_text=None):
    try:
        return parse_model_output(raw_text)
    except ValueError:
        if repair_text is None:
            raise
        try:
            return parse_model_output(repair_text)
        except ValueError:
            return empty_receipt()


def run_pipeline(page_outputs):
    # page_outputs: list of {raw, text, repair?}
    grounded_pages = []
    before = []
    for page in page_outputs:
        try:
            parsed = extract_one_page(page["raw"], page.get("repair"))
        except ValueError:
            parsed = empty_receipt()
        before.append(json.loads(json.dumps(parsed)))
        grounded_pages.append(ground_receipt(parsed, page.get("text") or ""))
    merged = merge_pages(grounded_pages)
    combined = "\n".join(page.get("text") or "" for page in page_outputs)
    grounded = ground_receipt(merged, combined)
    return check_totals(grounded), before, combined


print("pipeline ready")


## 4. CPU demo (no model)

A clerk-style receipt as text. The dummy “model” emits JSON (and one broken JSON to show repair). The pipeline must still return schema-valid output.


In [ ]:
PAGE_TEXT = '''Oak & Ember Cafe
15/03/2026
Latte  2 x 4.50  9.00
Croissant  1 x 3.50  3.50
SUBTOTAL 12.50
TAX 1.00
TOTAL 13.50
'''

good = json.dumps({
    "store_name": "Oak & Ember Cafe",
    "date": "2026-03-15",
    "line_items": [
        {"name": "Latte", "qty": 2, "unit_price": 4.5, "amount": 9.0},
        {"name": "Croissant", "qty": 1, "unit_price": 3.5, "amount": 3.5},
    ],
    "subtotal": 12.5,
    "tax": 1.0,
    "total": 13.5,
})
hallucinated = json.dumps({
    "store_name": "Totally Fake Shop",
    "date": "2026-03-15",
    "line_items": [{"name": "Latte", "qty": 2, "unit_price": 4.5, "amount": 9.0}],
    "subtotal": 12.5,
    "tax": 1.0,
    "total": 99.0,
})

ok, _, _ = run_pipeline([{"raw": good, "text": PAGE_TEXT}])
print("clean extract")
print(json.dumps(ok, indent=2))

broken, _, _ = run_pipeline([{"raw": "{not json", "repair": good, "text": PAGE_TEXT}])
print("after one repair, store =", broken["store_name"])

ghost, before, _ = run_pipeline([{"raw": hallucinated, "text": PAGE_TEXT}])
print("hallucinated store before grounding:", before[0]["store_name"])
print("after grounding:", ghost["store_name"], ghost["_verification"]["store_name"])
print("total kept but unverified because 9+0 != 99:", ghost["total"], ghost["_verification"]["total"])


## 5. Metrics (used for zero-shot and fine-tuned)


In [ ]:
def values_equal(left, right):
    if left is None or right is None:
        return left is None and right is None
    if isinstance(left, str) or isinstance(right, str):
        return normalize_text(str(left)) == normalize_text(str(right))
    try:
        return abs(float(left) - float(right)) <= SANITY_TOLERANCE
    except (TypeError, ValueError):
        return left == right


def prf(tp, fp, fn, exact, n):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "exact_match": exact / n if n else 0.0}


def field_scores(golds, preds, field):
    tp = fp = fn = exact = 0
    for gold, pred in zip(golds, preds):
        gv, pv = gold.get(field), pred.get(field)
        if gv is None and pv is None:
            exact += 1
        elif gv is None:
            fp += 1
        elif pv is None:
            fn += 1
        elif values_equal(gv, pv):
            tp += 1
            exact += 1
        else:
            fp += 1
            fn += 1
    return prf(tp, fp, fn, exact, len(golds))


def line_item_scores(golds, preds):
    totals = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0, "exact": 0, "n": 0})
    for gold, pred in zip(golds, preds):
        remaining = list(pred.get("line_items") or [])
        pairs = []
        for item in gold.get("line_items") or []:
            hit = None
            for i, cand in enumerate(remaining):
                if normalize_text(cand.get("name") or "") == normalize_text(item.get("name") or ""):
                    hit = remaining.pop(i)
                    break
            pairs.append((item, hit))
        pairs.extend((None, extra) for extra in remaining)
        for g_item, p_item in pairs:
            for leaf in ("name", "qty", "unit_price", "amount"):
                b = totals[leaf]
                b["n"] += 1
                gv = None if g_item is None else g_item.get(leaf)
                pv = None if p_item is None else p_item.get(leaf)
                if gv is None and pv is None:
                    b["exact"] += 1
                elif gv is None:
                    b["fp"] += 1
                elif pv is None:
                    b["fn"] += 1
                elif values_equal(gv, pv):
                    b["tp"] += 1
                    b["exact"] += 1
                else:
                    b["fp"] += 1
                    b["fn"] += 1
    return {leaf: prf(v["tp"], v["fp"], v["fn"], v["exact"], v["n"]) for leaf, v in totals.items()}


def exact_receipt(gold, pred):
    if any(not values_equal(gold.get(f), pred.get(f)) for f in ("store_name", "date", "subtotal", "tax", "total")):
        return False
    g = sorted(gold.get("line_items") or [], key=lambda i: normalize_text(i.get("name") or ""))
    p = sorted(pred.get("line_items") or [], key=lambda i: normalize_text(i.get("name") or ""))
    if len(g) != len(p):
        return False
    return all(values_equal(a.get(k), b.get(k)) for a, b in zip(g, p) for k in ("name", "qty", "unit_price", "amount"))


def hallucination_rate(preds, texts):
    missing = non_null = 0
    for pred, text in zip(preds, texts):
        values = [pred.get("store_name"), pred.get("date"), pred.get("subtotal"), pred.get("tax"), pred.get("total")]
        for item in pred.get("line_items") or []:
            values.extend([item.get("name"), item.get("qty"), item.get("unit_price"), item.get("amount")])
        for value in values:
            if value is None:
                continue
            non_null += 1
            if isinstance(value, str) and not text_present(value, text):
                missing += 1
            elif isinstance(value, (int, float)) and not numbers_match(float(value), text):
                missing += 1
    return 0.0 if not non_null else missing / non_null


def latency_percentiles(samples_ms):
    if not samples_ms:
        return {"p50": None, "p95": None}
    ordered = sorted(samples_ms)

    def pct(p):
        return ordered[min(len(ordered) - 1, max(0, int(round((p / 100) * (len(ordered) - 1)))))]

    return {"p50": pct(50), "p95": pct(95)}


def summarize(golds, preds, texts, latencies, stage):
    fields = {name: field_scores(golds, preds, name) for name in ("store_name", "date", "subtotal", "tax", "total")}
    fields["line_items"] = line_item_scores(golds, preds)
    exact = sum(exact_receipt(g, p) for g, p in zip(golds, preds)) / max(1, len(golds))
    peak = "n/a"
    if HAS_GPU and torch is not None:
        peak = f"{torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB"
    return {
        "stage": stage,
        "n": len(golds),
        "fields": fields,
        "exact_match_receipt": exact,
        "json_validity": 1.0,
        "hallucination_after_grounding": hallucination_rate(preds, texts),
        "latency_ms_per_page": latency_percentiles(latencies),
        "peak_gpu_memory": peak,
    }


def show_table(stage):
    print(f"\n## {stage['stage']} n={stage['n']}")
    print("| Field | P | R | F1 | Exact |")
    print("|---|---:|---:|---:|---:|")
    for name, scores in stage["fields"].items():
        if name == "line_items":
            for leaf, s in scores.items():
                print(f"| line_items.{leaf} | {s['precision']:.3f} | {s['recall']:.3f} | {s['f1']:.3f} | {s['exact_match']:.3f} |")
        else:
            print(f"| {name} | {scores['precision']:.3f} | {scores['recall']:.3f} | {scores['f1']:.3f} | {scores['exact_match']:.3f} |")
    lat = stage["latency_ms_per_page"]
    print(f"exact={stage['exact_match_receipt']:.3f}  hallu={stage['hallucination_after_grounding']:.3f}  "
          f"p50/p95={lat['p50']}/{lat['p95']}  gpu={stage['peak_gpu_memory']}")


print("metrics ready")


## 6. CORD v2 — official splits, frozen ids

Test ids are never used for training. If `datasets` is missing, this cell skips.


In [ ]:
dataset = None
splits = {"train": [], "validation": [], "test": []}
try:
    from datasets import load_dataset

    dataset = load_dataset("naver-clova-ix/cord-v2")
    for name in dataset:
        ids = []
        for index, row in enumerate(dataset[name]):
            gt = json.loads(row["ground_truth"]) if isinstance(row["ground_truth"], str) else row["ground_truth"]
            ids.append(str((gt.get("meta") or {}).get("image_id", f"{name}-{index}")))
        splits[name] = ids
    (WORK / "splits.json").write_text(
        json.dumps({"splits": splits, "hash": {k: split_hash(v) for k, v in splits.items()}}, indent=2),
        encoding="utf-8",
    )
    print({k: len(v) for k, v in splits.items()})
    print(json.dumps(cord_to_schema(ground_truth_of(dataset["train"][0])), indent=2)[:600])
except Exception as exc:
    print("CORD not loaded (install datasets or run on Colab):", exc)


In [ ]:
def wandb_init(extra=None):
    if os.environ.get("WANDB_MODE") == "disabled" or not os.environ.get("WANDB_API_KEY"):
        print("wandb disabled")
        return None
    try:
        import wandb

        cfg = {
            "model_id": MODEL_ID,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "lora_dropout": LORA_DROPOUT,
            "bits": 4,
            "image_resolution": LONG_EDGE,
            "learning_rate": 2e-4,
            "train_size": len(splits.get("train", [])),
            "val_size": len(splits.get("validation", [])),
            "split_hash": {k: split_hash(v) for k, v in splits.items() if v},
        }
        if extra:
            cfg.update(extra)
        return wandb.init(project=os.environ.get("WANDB_PROJECT", "vlm-receipt-extraction"), config=cfg)
    except Exception as exc:
        print("wandb skipped", exc)
        return None


wb = wandb_init()


## 7. VLM load + one-page infer (needs GPU)

Skip on CPU. Temperature 0. One repair generation if JSON/schema fails.


In [ ]:
vlm = {"model": None, "processor": None}


def load_vlm(adapter_path=None):
    if not HAS_GPU:
        print("no GPU — skip VLM load")
        return
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

    processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256 * 28 * 28, max_pixels=MAX_PIXELS)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, quantization_config=quant, device_map="auto", torch_dtype=torch.float16
    )
    if adapter_path and Path(adapter_path).exists():
        from peft import PeftModel

        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    vlm["model"], vlm["processor"] = model, processor
    print("loaded", MODEL_ID, "adapter", adapter_path or "base")


def infer_page(image, repair_hint=None):
    from PIL import Image as PILImage

    if vlm["model"] is None:
        raise RuntimeError("model not loaded")
    if not isinstance(image, PILImage.Image):
        image = image.convert("RGB") if hasattr(image, "convert") else PILImage.fromarray(image).convert("RGB")
    else:
        image = image.convert("RGB")
    w, h = image.size
    scale = LONG_EDGE / max(w, h)
    if scale < 1:
        image = image.resize((max(1, int(w * scale)), max(1, int(h * scale))))
    prompt = EXTRACT_PROMPT if not repair_hint else f"Previous output failed: {repair_hint}\n{EXTRACT_PROMPT}"
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
    text = vlm["processor"].apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = vlm["processor"](text=[text], images=[image], padding=True, return_tensors="pt")
    inputs = inputs.to(vlm["model"].device)
    out = vlm["model"].generate(**inputs, max_new_tokens=1024, do_sample=False, temperature=None)
    trimmed = out[:, inputs["input_ids"].shape[1] :]
    return vlm["processor"].batch_decode(trimmed, skip_special_tokens=True)[0]


def extract_image(image, page_text=""):
    raw = infer_page(image)
    repair = None
    try:
        parse_model_output(raw)
    except ValueError as exc:
        repair = infer_page(image, repair_hint=str(exc))
    result, before, _ = run_pipeline([{"raw": raw, "repair": repair, "text": page_text}])
    return result, raw, before[0]


if HAS_GPU:
    load_vlm()
else:
    print("CPU: VLM cells below will no-op")


## 8. Zero-shot eval on a frozen test slice

Same ids will be reused after fine-tuning. `EVAL_N` defaults to 16 so a T4 finishes.


In [ ]:
def cord_page_text(row):
    raw = row["ground_truth"]
    payload = json.loads(raw) if isinstance(raw, str) else raw
    lines = []
    for line in payload.get("valid_line") or []:
        words = [w.get("text", "") for w in line.get("words") or []]
        if words:
            lines.append(" ".join(words))
    return "\n".join(lines)


def cord_eval_rows(n=EVAL_N):
    if dataset is None:
        return []
    rows = []
    for row in dataset["test"]:
        gold = cord_to_schema(ground_truth_of(row))
        rows.append({"image": row["image"], "gold": gold, "text": cord_page_text(row)})
        if len(rows) >= n:
            break
    return rows


def eval_vlm(rows, stage):
    golds, preds, texts, latencies = [], [], [], []
    if vlm["model"] is None:
        print("no model — skip", stage)
        return None
    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
    for row in rows:
        started = time.perf_counter()
        result, raw, _ = extract_image(row["image"], row.get("text") or "")
        latencies.append((time.perf_counter() - started) * 1000.0)
        pred = {k: result[k] for k in ("store_name", "date", "line_items", "subtotal", "tax", "total")}
        golds.append(row["gold"])
        preds.append(pred)
        texts.append(row.get("text") or "")
        del raw
    summary = summarize(golds, preds, texts, latencies, stage)
    show_table(summary)
    return summary


eval_rows = cord_eval_rows()
print("eval rows", len(eval_rows))
zero_shot = eval_vlm(eval_rows, "zero-shot") if eval_rows and HAS_GPU else None


## 9. QLoRA fine-tune (needs GPU)

Language / attention / MLP only — no vision-encoder LoRA (T4 headroom). One epoch on official CORD train, or `MAX_TRAIN` to smoke-test.


In [ ]:
MAX_TRAIN = int(os.environ.get("MAX_TRAIN", "0"))  # 0 = full official train split


def to_messages(target):
    return [
        {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": EXTRACT_PROMPT}]},
        {"role": "assistant", "content": [{"type": "text", "text": json.dumps(target, ensure_ascii=False)}]},
    ]


def convert_split(split, limit=0):
    rows = []
    for row in split:
        rows.append({"messages": to_messages(cord_to_schema(ground_truth_of(row))), "image": row["image"]})
        if limit and len(rows) >= limit:
            break
    return rows


def train_unsloth(train_rows):
    from trl import SFTConfig, SFTTrainer
    from unsloth import FastVisionModel
    from unsloth.trainer import UnslothVisionDataCollator

    model, tokenizer = FastVisionModel.from_pretrained(MODEL_ID, load_in_4bit=True, use_gradient_checkpointing="unsloth")
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=False,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
    FastVisionModel.for_training(model)
    report = "wandb" if os.environ.get("WANDB_API_KEY") and os.environ.get("WANDB_MODE") != "disabled" else "none"
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer),
        train_dataset=train_rows,
        args=SFTConfig(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            num_train_epochs=1,
            learning_rate=2e-4,
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir=str(ADAPTER_DIR),
            report_to=report,
            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_length=2048,
        ),
    )
    trainer.train()
    ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    return ADAPTER_DIR


def train_hf(train_rows):
    from peft import LoraConfig, get_peft_model
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
    from trl import SFTConfig, SFTTrainer

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=quant, device_map="auto")
    model = get_peft_model(
        model,
        LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            bias="none",
            task_type="CAUSAL_LM",
        ),
    )
    report = "wandb" if os.environ.get("WANDB_API_KEY") else "none"
    trainer = SFTTrainer(
        model=model,
        processing_class=processor,
        train_dataset=train_rows,
        args=SFTConfig(
            per_device_train_batch_size=1,
            gradient_accumulation_steps=8,
            num_train_epochs=1,
            learning_rate=2e-4,
            logging_steps=1,
            output_dir=str(ADAPTER_DIR),
            report_to=report,
            remove_unused_columns=False,
        ),
    )
    trainer.train()
    model.save_pretrained(ADAPTER_DIR)
    processor.save_pretrained(ADAPTER_DIR)
    return ADAPTER_DIR


trained = False
if HAS_GPU and dataset is not None:
    train_rows = convert_split(dataset["train"], MAX_TRAIN)
    print("train rows", len(train_rows), "test held out", len(splits["test"]))
    try:
        train_unsloth(train_rows)
    except Exception as exc:
        print("Unsloth failed, falling back:", exc)
        train_hf(train_rows)
    trained = True
    if HAS_GPU:
        print("peak GPU GiB", torch.cuda.max_memory_allocated() / 1024**3)
else:
    print("skip training — need GPU + CORD")


## 10. Fine-tuned eval, comparison, optional Hub push

Reload the adapter and score the **same** `eval_rows` as zero-shot.


In [ ]:
fine_tuned = None
if trained and HAS_GPU:
    vlm["model"] = vlm["processor"] = None
    load_vlm(ADAPTER_DIR)
    fine_tuned = eval_vlm(eval_rows, "fine-tuned")

stages = [s for s in (zero_shot, fine_tuned) if s]
report = {
    "note": "Standalone notebook 03. Test ids never used in training.",
    "stages": stages,
    "hardware": "T4" if HAS_GPU else "CPU (VLM stages skipped)",
}
(WORK / "metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

lines = ["# Evaluation (standalone notebook)", "", report["note"], ""]
for stage in stages:
    show_table(stage)
if not stages:
    print("No VLM stages. CPU demo above still ran the full post-process pipeline.")
print("wrote", WORK / "metrics.json")

if os.environ.get("WANDB_API_KEY") and stages:
    try:
        import wandb

        run = wandb.run or wandb.init(project=os.environ.get("WANDB_PROJECT", "vlm-receipt-extraction"))
        table = wandb.Table(columns=["stage", "field", "f1", "exact"])
        for stage in stages:
            for name, scores in stage["fields"].items():
                if name == "line_items":
                    continue
                table.add_data(stage["stage"], name, scores["f1"], scores["exact_match"])
        run.log({"comparison": table})
        wandb.finish()
    except Exception as exc:
        print("wandb log skipped", exc)

if os.environ.get("HF_PUSH") == "1" and ADAPTER_DIR.exists():
    from huggingface_hub import HfApi

    api = HfApi()
    api.create_repo(HF_ADAPTER_REPO, exist_ok=True, private=True)
    api.upload_folder(folder_path=str(ADAPTER_DIR), repo_id=HF_ADAPTER_REPO)
    print("pushed", HF_ADAPTER_REPO)
else:
    print("adapter local at", ADAPTER_DIR, "— set HF_PUSH=1 to upload to", HF_ADAPTER_REPO)


## How to run

| Environment | What runs |
|---|---|
| This laptop, no GPU | Cells 0–6: schema, grounding, sanity, merge, CPU demo |
| Colab T4 | Everything. Optional: `EVAL_N=32`, `MAX_TRAIN=0` (full train), `WANDB_API_KEY`, `HF_PUSH=1` |

Do not open the official CORD **test** split for prompt edits. The eval cells read it only after the split hash is frozen.
